In [1]:
import copy
import dataclasses

import numpy
import scipy.spatial

import cloudvolume
import kimimaro
import fastremap

np = numpy

In [2]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

from ac_pcg.pcgraph.edges import Edges
from ac_pcg.pcgraph.edges import EDGE_TYPES

from ac_pcg.io.edges import put_chunk_edges
from ac_pcg.io.components import put_chunk_components

from ac_pcg.utils import (
    label_chunk,
    chunk_edges_from_skeleton
)

In [3]:
import gzip
import pathlib
import pickle

def read_gzip_array(fn, preprocess_func=lambda x: x):
    with gzip.open(fn, "rb") as f:
        a = numpy.load(f)
    return preprocess_func(a)

test_data_path = pathlib.Path(
    "/allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/pcg_axconn/test_data_strip/"
)

test_data_labeled_array_path = test_data_path / "H17_x55_S32_230412_Pos42.npy.gz"
test_skels_path = test_data_path / "H17_x55_S32_230412_Pos42.skels.pkl"

In [4]:
labeled_array = read_gzip_array(test_data_labeled_array_path)
with test_skels_path.open(mode="rb") as skels_fobj:
    label_skels = pickle.load(skels_fobj)

In [5]:
%%time
import rtree

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}


p = rtree.index.Property()
p.dimension = 3
label_skel_idx = rtree.index.Index(
    ((sk_id, skel_bb.to_list(), label_skels[sk_id]) for sk_id, skel_bb in skel_id_to_bboxes.items()),
    properties=p
)

CPU times: user 3.45 s, sys: 50.8 ms, total: 3.5 s
Wall time: 3.53 s


In [6]:
labeled_array_chunksize = numpy.array((128, 128, 128))

In [7]:
def get_bbox_chunks(bbox, chunk_size, offset=None):
    if offset is not None:
        raise NotImplementedError
    (chunk_min, chunk_max), (remainder_min, remainder_max) = numpy.divmod(
        numpy.array([bbox.minpt, bbox.maxpt]), chunk_size)
    # chunk_max += remainder_max.astype(bool)

    # TODO could be more clever about dims
    imin, jmin, kmin = chunk_min
    imax, jmax, kmax = chunk_max
    chunks = np.mgrid[imin:imax+1:1, jmin:jmax+1:1, kmin:kmax+1:1].reshape(3, -1).T

    return chunks

In [8]:
%%time

skel_to_chunks = {sk_id: get_bbox_chunks(sk_bbox, labeled_array_chunksize) for sk_id, sk_bbox in skel_id_to_bboxes.items()}

chunk_to_skel_ids = {}
for sk_id, sk_chunks in skel_to_chunks.items():
    for sk_chunk in sk_chunks:
        try:
            chunk_to_skel_ids[tuple(sk_chunk)].append(sk_id)
        except KeyError:
            chunk_to_skel_ids[tuple(sk_chunk)] = [sk_id]

CPU times: user 1.67 s, sys: 3.22 ms, total: 1.67 s
Wall time: 1.66 s


In [9]:
def chunk_idx_to_bbox(chunk_idx, chunk_size, chunked_box_shape):
    chunk_mins = tuple(idx * chunk_d for idx, chunk_d in zip(chunk_idx, chunk_size))
    chunk_max = tuple(min(box_d, chunk_min + chunk_d) for chunk_min, chunk_d, box_d in zip(chunk_mins, chunk_size, chunked_box_shape))
    bbox = cloudvolume.Bbox(chunk_mins, chunk_max)
    return ac_pcg.chunks.ChunkBbox(
        bbox=bbox,
        chunk_idx=chunk_idx
    )


def process_oversegment_array(arr, skels, label_func, oversegment_kwargs=None, relabel_zero=False):
    oversegment_kwargs = oversegment_kwargs or {}

    oversegmented_arr, oversegmented_skels = kimimaro.utility.oversegment(
        arr, skels, **oversegment_kwargs)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: label_func(input_lbl)  # input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_arr
        )
    }
    if not relabel_zero:
        lbl_map[0] = 0

    relabeled_arr = fastremap.remap(oversegmented_arr, lbl_map)

    for skel in oversegmented_skels:
        skel.segments = fastremap.remap(skel.segments, lbl_map)

    return relabeled_arr, oversegmented_skels


def process_oversegment_bbox_shmem(bbox, sharedarray_props, *args, **kwargs):
    existing_shm = multiprocessing.shared_memory.SharedMemory(name=sharedarray_props.sharedmem_name)
    arr = numpy.ndarray(sharedarray_props.shape, dtype=sharedarray_props.dtype, buffer=existing_shm.buf)
    bbox_arr = arr[bbox.to_slices()]
    
    result = process_oversegment_array(arr, *args, **kwargs)
    existing_shm.close()
    return result

In [10]:
# run oversegmentation over all chunks

import time

chunk_size = labeled_array_chunksize  # (128, 128, 128)
chunk_boxes = ac_pcg.chunks.iterate_chunk_slice_boxes(
    labeled_array.shape, chunk_size)

labeler = ac_pcg.label.ChunkLabeler()

output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

oversegment_kwargs = {
    "downsample": 6,
    "progress": False
}

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(res.object, chunk_contains_bb)
        for res in label_skel_idx.intersection(
            chunk_contains_bb.to_list(), objects=True
        )
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    oversegmented_subvol_arr, oversegmented_subvol_skels = process_oversegment_array(
        subvol_arr, subvol_skels, lambda x: labeler.encode_chunk_seg(chunk_box.chunk_idx, x),
        oversegment_kwargs=oversegment_kwargs, relabel_zero=False
    )

    # oversegmented_subvol_arr, oversegmented_subvol_skels = kimimaro.utility.oversegment(
    #     subvol_arr, subvol_skels, downsample=6, progress=False)

    # # convert labels to uint64 layer/chunk indices
    # lbl_map = {
    #     input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
    #     for input_lbl in fastremap.unique(
    #         oversegmented_subvol_arr
    #     ) if input_lbl != 0
    # }
    # relabeled_arr = fastremap.remap(oversegmented_subvol_arr, lbl_map)
    
    output_arr[chunk_box.bbox.to_slices()] = oversegmented_subvol_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments

    # # map new indices to original skel vertices
    # for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
    #     output_skel = output_skels[skel.id]
    #     skel.segments = fastremap.remap(skel.segments, lbl_map)
    #     try:
    #         output_skel.segments[subvol_indices] = skel.segments
    #     except AttributeError:
    #         output_skel.add_vertex_attribute(
    #             "segments",
    #             numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
    #         )
    #         output_skel.segments[subvol_indices] = skel.segments
    

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

print(time.time() - tic)

90 0.26351475715637207
100 0.7832183837890625
120 2.5808541774749756
130 3.74359393119812
140 4.7852442264556885
150 5.926273822784424
160 6.994014263153076
170 8.103637456893921
180 9.426310300827026
190 10.850332736968994
200 11.99742865562439
210 13.238438367843628
220 14.693318605422974
230 15.978116035461426
240 17.302445888519287
250 18.70110273361206
260 19.97762393951416
270 21.731225967407227
280 23.695383310317993
290 25.516634225845337
300 27.444515705108643
310 29.331429481506348
320 30.766585111618042
330 32.12680673599243
340 33.326746463775635
350 34.499146938323975
360 36.067567348480225
370 37.73861002922058
390 40.450578451156616
400 42.389726400375366
420 45.24803042411804
430 46.62195134162903
450 49.53513765335083
460 51.08215928077698
480 53.75109553337097
490 55.05523061752319
500 56.2459511756897
510 57.452192544937134
520 58.729268074035645
530 59.85082125663757
540 61.22609996795654
550 62.532082080841064
560 63.587944746017456
570 64.63332533836365
580 65.756

In [11]:
%%time

@dataclasses.dataclass
class VtxIdxLoc:
    vertices: numpy.ndarray
    indices: numpy.ndarray
    locations: numpy.ndarray


def skeleton_unique_vtxs_idxs_locs(skel):
    vtxs, idxs = fastremap.unique(skel.segments, return_index=True)
    locs = skel.vertices[idxs]
    return VtxIdxLoc(vtxs, idxs, locs)


# skel_id_to_unique_vtxs_idxs_locs = {
#     skel_id: skeleton_unique_vtxs_idxs_locs(skel) for skel_id, skel in output_skels.items()
# }

CPU times: user 435 μs, sys: 10 μs, total: 445 μs
Wall time: 452 μs


In [12]:
%%time

skel_id_to_unique_vtxs_idxs_locs = {}
skel_id_to_full_array_idxs = {}

offset = 0

for skel_id, skel in output_skels.items():
    skel_id_to_unique_vtxs_idxs_locs[skel_id] = skeleton_unique_vtxs_idxs_locs(skel)
    vtx_size = skel_id_to_unique_vtxs_idxs_locs[skel_id].vertices.size
    skel_id_to_full_array_idxs[skel_id] = numpy.arange(offset, offset + vtx_size)
    offset += vtx_size
    

CPU times: user 3.76 s, sys: 28 ms, total: 3.79 s
Wall time: 3.79 s


In [13]:
%%time

all_vtxs, all_idxs, all_locs = zip(*((v.vertices, v.indices, v.locations) for k, v in skel_id_to_unique_vtxs_idxs_locs.items()))

all_vtxs = numpy.concatenate(all_vtxs)
all_idxs = numpy.concatenate(all_idxs)
all_locs = numpy.concatenate(all_locs)

all_kdtree = scipy.spatial.KDTree(all_locs)

CPU times: user 203 ms, sys: 11.9 ms, total: 215 ms
Wall time: 209 ms


In [14]:
%%time
# generate edges between skeletons enforcing non-directional uniqueness

distance = 30
edge_set = set()

for i, (skel_id, vtxs_idxs_locs) in enumerate(skel_id_to_unique_vtxs_idxs_locs.items()):
    skel_tree = scipy.spatial.KDTree(vtxs_idxs_locs.locations)
    pairs = skel_tree.query_ball_tree(all_kdtree, r=distance)
    in_skel_ids = set(skel_id_to_full_array_idxs[skel_id])

    for skel_idx, pair_result in enumerate(pairs):
        skel_vtx = vtxs_idxs_locs.vertices[skel_idx]
        edge_set |= {
            frozenset((skel_vtx, all_vtxs[query_vtx_idx]))
            for query_vtx_idx in (set(pair_result) - in_skel_ids)
            if (skel_vtx != all_vtxs[query_vtx_idx])
        }
    if not i % 1000:
        print(i)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
CPU times: user 34.3 s, sys: 985 ms, total: 35.3 s
Wall time: 35.2 s


In [15]:
%time all_pairs = numpy.array([tuple(edge) for edge in edge_set])
all_pairs.shape

CPU times: user 9.75 s, sys: 380 ms, total: 10.1 s
Wall time: 10.1 s


(7942175, 2)

In [16]:
# %%time
# # generate edges
# 
# distance = 30
# all_pairs =  []
# 
# for i, (skel_id, vtxs_idxs_locs) in enumerate(skel_id_to_unique_vtxs_idxs_locs.items()):
#     skel_tree = scipy.spatial.KDTree(vtxs_idxs_locs.locations)
#     pairs = skel_tree.query_ball_tree(all_kdtree, r=distance)
#     for skel_idx, pair_result in enumerate(pairs):
#         vtx = vtxs_idxs_locs.vertices[skel_idx]
#         pair_result = numpy.array(pair_result)
#         pair_result = pair_result[
#            ~numpy.isin(pair_result, skel_id_to_full_array_idxs[skel_id])
#         ]
#         pair_result = numpy.c_[all_vtxs[pair_result], numpy.full(pair_result.shape, vtx)]
#         all_pairs.append(pair_result)
#     if not i % 1000:
#         print(i)

In [17]:
# %time all_pairs = numpy.concatenate(all_pairs)
# %time all_pairs = numpy.sort(all_pairs, axis=1)
# %time all_pairs = fastremap.unique(all_pairs, axis=0)

In [18]:
# import fixes...
import importlib
importlib.reload(ac_pcg.utils)
importlib.reload(ac_pcg.pcgraph.edges)
importlib.reload(ac_pcg.pcgraph)

<module 'ac_pcg.pcgraph' (<_frozen_importlib_external.NamespaceLoader object at 0x7f54ad124550>)>

In [19]:
@dataclasses.dataclass
class ChunkEdgeComponentResult:
    in_chunk_edges: numpy.ndarray
    between_chunk_edges: numpy.ndarray
    chunk_components: numpy.ndarray


@dataclasses.dataclass
class ChunkEdgeResult:
    in_chunk_edges: numpy.ndarray
    between_chunk_edges: numpy.ndarray


def filter_edges_by_chunk(edge_array, query_chunk, labeler):
    segid_bits = labeler.segid_bits
    query_layer_chunk_idx = label_chunk(labeler, query_chunk)

    layer_chunk_edges = edge_array >> segid_bits
    query_chunk_edges_mask = (layer_chunk_edges == query_layer_chunk_idx)
    in_chunk_edges_mask = query_chunk_edges_mask[:, 0] & query_chunk_edges_mask[:, 1]
    between_chunk_edges_mask = query_chunk_edges_mask[:, 0] ^ query_chunk_edges_mask[:, 1]

    in_chunk_edges = edge_array[in_chunk_edges_mask]
    between_chunk_edges = edge_array[between_chunk_edges_mask]

    return ChunkEdgeResult(
        in_chunk_edges=in_chunk_edges,
        between_chunk_edges=between_chunk_edges
    )


def filter_edge_arr_by_vtx(edge_array, vtx):
    return edge_array[
        numpy.all(edge_array != vtx, axis=1)
    ]


def chunk_edges_components_from_skeleton(skel, query_chunk, labeler):
    seg_edges = skel.segments[skel.edges]

    reduced_seg_edges = fastremap.unique(
        numpy.sort(
            seg_edges[
            (seg_edges[:, 0] != seg_edges[:, 1])
            ], axis=1
        ),
        axis=0)

    # NOTE -- there are zeros in components and edges -- filter from full edge list here
    reduced_seg_edges = filter_edge_arr_by_vtx(reduced_seg_edges, 0)

    segid_bits = labeler.segid_bits

    # calculate layer and chunk label for edges
    layer_chunk_edges = reduced_seg_edges >> segid_bits

    # calculate chunk label and find occurences
    query_layer_chunk_idx = label_chunk(labeler, query_chunk)
    query_chunk_edges_mask = (layer_chunk_edges == query_layer_chunk_idx)

    in_chunk_edges_mask = query_chunk_edges_mask[:, 0] & query_chunk_edges_mask[:, 1]
    between_chunk_edges_mask = query_chunk_edges_mask[:, 0] ^ query_chunk_edges_mask[:, 1]

    in_chunk_edges = reduced_seg_edges[in_chunk_edges_mask]
    between_chunk_edges = reduced_seg_edges[between_chunk_edges_mask]

    unique_components = fastremap.unique(numpy.concatenate((in_chunk_edges, between_chunk_edges), dtype=numpy.int64))
    chunk_components = (unique_components if unique_components.size else None)
    # if unique_components.size:
    #     chunk_components = numpy.concatenate(([unique_components.size], unique_components), dtype=numpy.int64)
    # else:
    #     chunk_components = None

    return ChunkEdgeComponentResult(
        in_chunk_edges=in_chunk_edges,
        between_chunk_edges=between_chunk_edges,
        chunk_components=chunk_components)

In [20]:
%%time
edge_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/edges"
component_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/components"

 # NOTE -- there are zeros in components and edges -- filter from full edge list here
all_pairs = filter_edge_arr_by_vtx(all_pairs, 0)

tic = time.time()

for chunk_num, (chunk, chunk_skel_ids) in enumerate(chunk_to_skel_ids.items()):
    in_chunk_edge_results = []
    between_chunk_edge_results = []
    connected_components_results = []
    # get connected components and active edges for all skeletons
    for skel_id in chunk_skel_ids:
        skel = output_skels[skel_id]
        chunk_edge_components = chunk_edges_components_from_skeleton(skel, chunk, labeler)

        if chunk_edge_components.in_chunk_edges is not None:
            in_chunk_edge_results.append(chunk_edge_components.in_chunk_edges)
        if chunk_edge_components.between_chunk_edges is not None:
            between_chunk_edge_results.append(chunk_edge_components.between_chunk_edges)
        if chunk_edge_components.chunk_components is not None:
            connected_components_results.append(chunk_edge_components.chunk_components)

   

    # get inactive edges from spatial query results
    chunk_inactive_edges = filter_edges_by_chunk(all_pairs, chunk, labeler)

    in_chunk_edge_results.append(chunk_inactive_edges.in_chunk_edges)
    between_chunk_edge_results.append(chunk_inactive_edges.between_chunk_edges)
    
    in_chunk_edge_arr = numpy.concatenate(in_chunk_edge_results)
    between_chunk_edge_arr = numpy.concatenate(between_chunk_edge_results)

    in_chunk_edges = Edges(*in_chunk_edge_arr.T)
    between_chunk_edges = Edges(*between_chunk_edge_arr.T)
    # cross_chunk_edges are not generated in this segmentation method
    cross_chunk_edges = Edges([], [])

    # produce chunk edge format
    edges_d = {
        EDGE_TYPES.in_chunk: in_chunk_edges,
        EDGE_TYPES.between_chunk: between_chunk_edges,
        EDGE_TYPES.cross_chunk: cross_chunk_edges
    }

    # serialization handles writing connected components format
    connected_components = connected_components_results

    has_edges = len(in_chunk_edge_results) or len(between_chunk_edge_results)
    has_components = len(connected_components)

    if has_edges:
        put_chunk_edges(edge_output_loc, chunk, edges_d, compression_level=22)
    if has_components:
        put_chunk_components(component_output_loc, connected_components, chunk)

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

0 0.26430749893188477
10 2.6766459941864014
20 5.029514789581299
30 7.126909255981445
40 8.966349601745605
50 10.7893648147583
60 12.758227825164795
70 15.014114141464233
80 16.98209047317505
90 19.14235544204712
100 21.962758541107178
110 24.58846879005432
120 26.495809316635132
130 28.722872018814087
140 30.991816520690918
150 32.922504901885986
160 34.91175436973572
170 36.87961173057556
180 38.749308586120605
190 41.3052818775177
200 43.50539040565491
210 45.24046802520752
220 47.014888763427734
230 48.596230030059814
240 50.10318470001221
250 51.19459056854248
260 52.769673109054565
270 55.01026010513306
280 56.7414014339447
290 58.5872528553009
300 60.532500982284546
310 61.818475008010864
320 64.0938401222229
330 66.32126832008362
340 68.40785431861877
350 70.08004832267761
360 72.13446378707886
370 73.99693131446838
380 75.27583956718445
390 76.93874645233154
400 78.3199393749237
410 79.17218208312988
420 80.68520474433899
430 82.24044871330261
440 83.93193197250366
450 85.8791

In [21]:
label_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/labels"
cv_info = {
    "data_type": "uint64",
    "num_channels": 1,
    "scales": [
        {
            "chunk_sizes": [
                [
                    128,
                    128,
                    128
                ]
            ],
            "compressed_segmentation_block_size": (8, 8, 8),
            "encoding": "compressed_segmentation",
            "key": "1000_1000_1000",
            "resolution": [
                1000,
                1000,
                1000
            ],
            "size": output_arr.shape,
            "voxel_offset": [
                0,
                0,
                0
            ]
        }
    ],
    "type": "segmentation"
}

In [22]:
seg_cv = cloudvolume.CloudVolume(label_output_loc, info=cv_info)


In [23]:
seg_cv.commit_info()
seg_cv[..., 0] = output_arr[...]

Uploading: 100%|██████████████████████████████████████| 1521/1521 [00:44<00:00, 34.15it/s]
